# Managed MCP: Firestore Server Demo (February 2026 Preview)

This notebook demonstrates how to use the **Firestore Remote MCP (Model Context Protocol) Server** to allow an AI agent to interact with Firestore documents as 'tools'.

## Use Case
A customer support agent needs to retrieve and update user preferences stored in Firestore. Instead of writing custom API code, the agent uses the standard MCP protocol to 'discover' and 'use' Firestore as a tool.

### Requirements
- Firestore database enabled.
- MCP client (like Gemini CLI or ADK) configured with the Firestore MCP endpoint.

In [ ]:
!pip install google-genai-adk # ADK for agent development
from google.colab import auth
auth.authenticate_user()

import os
project_id = 'YOUR_PROJECT_ID'  # @param {type:"string"}
firestore_instance = '(default)' # @param {type:"string"}

### 2. Configure the ADK Agent with MCP Tool
The agent is told to use the Firestore MCP server.

In [ ]:
from vertexai.preview import agent_engine

# Conceptual: Define the MCP tool for Firestore
mcp_firestore_tool = agent_engine.MCPTool(
    name="firestore_search",
    endpoint="https://firestore.googleapis.com/mcp", # Example MCP endpoint
    config={
        "project": project_id,
        "database": firestore_instance
    }
)

agent = agent_engine.Agent(
    model="gemini-1.5-pro",
    tools=[mcp_firestore_tool],
    instruction="You are a customer support agent. Use Firestore to look up user data when asked."
)

print("Agent Initialized with Firestore MCP Tool")

### 3. Test the Agent
The agent will automatically decide to call the Firestore MCP tool to answer the user's question.

In [ ]:
user_input = "What is the current subscription status for user 'user_456'?"
response = agent.chat(user_input)

print(f"User: {user_input}")
print(f"Agent: {response.text}")

### 4. Things to remember or know
- **Unified Tooling**: MCP provides a standard way for agents to talk to ANY GCP service (Firestore, BigQuery, Monitoring) without custom SDK integration.
- **Faster Time-to-Market**: Developers don't need to write 'get_user_doc' or 'update_user_doc' functions; they just provide the MCP endpoint.
- **Enterprise Scalability**: Managed MCP servers handle authentication and quota automatically.